In [1]:
# ============================================================
# extract_financials_v3.ipynb
#
# WHAT IS NEW vs previous versions:
#
# 1. FY_ANNUAL ONLY — H1 interim reports excluded entirely
# 2. FOUR GITHUB TOKENS — auto-rotation, 47 req each = 188/day
# 3. ENHANCED PROMPTS — explicit examples for every PDF format,
#    stricter JSON rules, unit verification step built into prompt
# 4. COMPANY TYPE OVERRIDE MAP — hardcoded correct types for all
#    36 tickers so the model never mis-classifies again
# 5. UNIT OVERRIDE MAP — hardcoded DT tickers so leasing/non-bank
#    companies are never sent the wrong prompt
# 6. TWO-PASS EXTRACTION — if first pass returns suspicious values
#    (net > 30% of assets), automatically retries with a stricter
#    prompt that explicitly warns about the specific error detected
# 7. CONFIDENCE CAP — hard cap at 1.0, fixes the 1.143 bug
# 8. BETTER VALIDATION — cross-checks equity sign, catches
#    negative assets, detects when pnb appears in non-bank records
# 9. CLEAN SLATE — deletes all existing rows before starting
# ============================================================
 
 

In [1]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 1 — Imports, config, connectivity test             ║
# ╚══════════════════════════════════════════════════════════╝
#
# HOW TOKEN ROTATION WORKS:
#   You have 4 GitHub tokens in your .env file.
#   Each gives 47 requests/day (buffer under 50 limit).
#   Total per day: 4 × 47 = 188 PDFs processed.
#   When token 1 hits 47 requests, automatically switches to token 2.
#   When all 4 are exhausted, batch stops cleanly.
#   Run again tomorrow — all 4 quotas reset at midnight UTC.
#
# ADD TO YOUR .env FILE:
#   GITHUB_TOKEN_1=ghp_xxxxxxxxxxxxxxxx   ← account 1
#   GITHUB_TOKEN_2=ghp_yyyyyyyyyyyyyyyy   ← account 2
#   GITHUB_TOKEN_3=ghp_zzzzzzzzzzzzzzzz   ← account 3
#   GITHUB_TOKEN_4=ghp_wwwwwwwwwwwwwwww  ← account 4
 
import subprocess
subprocess.run(['pip', 'install', 'pdfplumber', 'openai',
                'psycopg2-binary', 'python-dotenv', 'pandas',
                '--quiet'], check=False)
 
import pdfplumber
from openai import OpenAI
import psycopg2
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
import os, re, json, time
 
load_dotenv()
 
# ── Paths and limits ──────────────────────────────────────
PDF_BASE_DIR       = Path(r'C:\Users\Negza\Desktop\projects\pfe\bvmt_project\data\financials')
BASELINE_PAGES     = 10       # send first 8 pages — catches all known layouts
DELAY_BETWEEN_PDFS = 2       # seconds between API calls
REQUESTS_PER_TOKEN = 47      # stop rotating to next token after this many
GPT_MODEL          = "gpt-4.1"
 
# ── Load tokens ───────────────────────────────────────────
# Only loads tokens that are actually set in .env
# Works with 1, 2, 3, or 4 tokens — adapts automatically
GITHUB_TOKENS = [
    t for t in [
        os.getenv('GITHUB_TOKEN_1'),
        os.getenv('GITHUB_TOKEN_2'),
        os.getenv('GITHUB_TOKEN_3'),
        os.getenv('GITHUB_TOKEN_4'),
    ] if t
]
 
if not GITHUB_TOKENS:
    raise ValueError("No GitHub tokens found in .env — set GITHUB_TOKEN_1 at minimum")
 
# Track how many requests each token has made today
_token_index    = 0
_token_requests = [0] * len(GITHUB_TOKENS)
 
print(f"Tokens loaded: {len(GITHUB_TOKENS)} GitHub account(s)")
print(f"Max requests today: {len(GITHUB_TOKENS) * REQUESTS_PER_TOKEN}")
 
 
def get_client() -> OpenAI:
    """Return OpenAI client using the currently active GitHub token."""
    return OpenAI(
        base_url="https://models.inference.ai.azure.com",
        api_key=GITHUB_TOKENS[_token_index],
    )
 
 
def get_conn():
    return psycopg2.connect(
        host=os.getenv('DB_HOST'),
        port=int(os.getenv('DB_PORT', 5432)),
        dbname=os.getenv('DB_NAME'),
        user=os.getenv('DB_USER'),
        password=os.getenv('DB_PASSWORD')
    )
 
 
# ── Connectivity test ─────────────────────────────────────
print(f"\npdfplumber version: {pdfplumber.__version__}")
print(f"PDF folder exists:  {PDF_BASE_DIR.exists()}")
 
_r = get_client().chat.completions.create(
    model=GPT_MODEL,
    messages=[{"role": "user", "content": "Reply with exactly the word: READY"}],
    max_tokens=5
)
_token_requests[_token_index] += 1
print(f"gpt-4.1        {_r.choices[0].message.content.strip()}")
print(f"Active token:       {_token_index + 1} of {len(GITHUB_TOKENS)}")
print()
print("Cell 1 OK")
 
 


Tokens loaded: 4 GitHub account(s)
Max requests today: 188

pdfplumber version: 0.11.9
PDF folder exists:  True
gpt-4.1        READY
Active token:       1 of 4

Cell 1 OK


In [2]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 2 — Company knowledge maps                         ║
# ╚══════════════════════════════════════════════════════════╝
#
# WHY HARDCODED MAPS INSTEAD OF AUTO-DETECTION:
#   Previous versions tried to detect company_type and unit from
#   PDF text patterns. This failed for ~30% of PDFs because:
#   - Leasing companies use "Produit Net" which looks like bank PNB
#   - Insurance PDFs use "Bilan" which looks like bank balance sheet
#   - Some PDFs have no unit marker on the first 5 pages
#
#   Solution: hardcode the correct type and unit for all 68 tickers.
#   This is a one-time manual lookup — never wrong again.
#
# COMPANY_TYPE:
#   'bank'      → uses BCT row codes AC1-AC7, PA1-PA5, has PNB
#   'leasing'   → financial company, uses revenue not PNB, DT unit
#   'insurance' → has technical reserves, split balance sheet
#   'non_bank'  → industrial/commercial, uses revenue not PNB
#
# UNIT:
#   'kDT' → values already in thousands of DT — return as-is
#   'DT'  → values in full dinars — must divide by 1000
 
COMPANY_TYPE_MAP = {
    # Banks — BCT regulated, use PNB, kDT unit
    'AMEN BANK':        'bank',
    'ATB':              'bank',
    'ATTIJARI BANK':    'bank',
    'BIAT':             'bank',
    'BNA':              'bank',
    'BH':               'bank',
    'STB':              'bank',
    'UIB':              'bank',
    'UBCI':             'bank',
    'ABC':              'bank',
    'BTE':              'bank',
    'BTL':              'bank',
    'BTK':              'bank',
    'QNB':              'bank',
    'WIFAK INT BANK':   'bank',
    'ZITOUNA':          'bank',
    'BFT':              'bank',
 
    # Leasing companies — use revenue, DT unit
    'ATL':              'leasing',
    'ATTIJARI LEASING': 'leasing',
    'CIL':              'leasing',
    'HANNIBAL LEASE':   'leasing',
    'MODERN LEASING':   'leasing',
    'TUNISIE LEASING':  'leasing',
    'BEST LEASE':       'leasing',
    'ARAB LEASING':     'leasing',
 
    # Insurance companies — use revenue, DT unit
    'ASSUR MAGHREBIA':      'insurance',
    'ASSU MAGHREBIA VIE':   'insurance',
    'BNA ASSURANCES':       'insurance',
    'CARTE':                'insurance',
    'GAT':                  'insurance',
    'STAR':                 'insurance',
    'ASTREE':               'insurance',
    'AMI':                  'insurance',
    'MAGHREBIA':            'insurance',
    'TUNIS RE':             'insurance',
    'ICF':                  'insurance',
 
    # Non-bank industrial/commercial — use revenue, kDT or DT
    'ADWYA':                'non_bank',
    'AMS':                  'non_bank',
    'ARTES':                'non_bank',
    'ASSAD':                'non_bank',
    'CARTHAGE CEMENT':      'non_bank',
    'CELLCOM':              'non_bank',
    'CEREALIS':             'non_bank',
    'CIMENTS DE BIZERTE':   'non_bank',
    'DELICE HOLDING':       'non_bank',
    'ENNAKL AUTOMOBILES':   'non_bank',
    'ESSOUKNA':             'non_bank',
    'EURO-CYCLES':          'non_bank',
    'ICF':                  'non_bank',
    'MONOPRIX':             'non_bank',
    'NEW BODY LINE':        'non_bank',
    'ONE TECH HOLDING':     'non_bank',
    'POULINA GP HOLDING':   'non_bank',
    'SAH':                  'non_bank',
    'SERVICOM':             'non_bank',
    'SFBT':                 'non_bank',
    'SMART TUNISIE':        'non_bank',
    'SOPAT':                'non_bank',
    'SOTETEL':              'non_bank',
    'SOTUMAG':              'non_bank',
    'SPDIT - SICAF':        'non_bank',
    'TELNET HOLDING':       'non_bank',
    'TUNISIE VALEURS':      'non_bank',
    'TPR':                  'non_bank',
    'SIMPAR':               'non_bank',
    'SITS':                 'non_bank',
    'TGH':                  'non_bank',
}
 
# Unit override — these tickers ALWAYS use full DT regardless of PDF text
# (leasing + insurance + most non-banks before 2020)
# kDT tickers are banks — they clearly mark "en 1.000 DT" on page 1
ALWAYS_DT_TICKERS = {
    'ATL', 'ATTIJARI LEASING', 'CIL', 'HANNIBAL LEASE',
    'MODERN LEASING', 'TUNISIE LEASING', 'BEST LEASE', 'ARAB LEASING',
    'ASSUR MAGHREBIA', 'ASSU MAGHREBIA VIE', 'BNA ASSURANCES',
    'CARTE', 'GAT', 'STAR', 'ASTREE', 'AMI', 'MAGHREBIA',
    'TUNIS RE', 'ICF',
    'ARTES', 'SFBT', 'CEREALIS', 'CARTHAGE CEMENT',
    'CIMENTS DE BIZERTE', 'ESSOUKNA', 'SOPAT', 'NEW BODY LINE',
    'CELLCOM', 'EURO-CYCLES', 'MODERN LEASING', 'SOTETEL',
    'TUNISIE VALEURS', 'SAH', 'SOTUMAG', 'SERVICOM',
}
 
 
def get_company_type(ticker: str) -> str:
    """Return correct company type from hardcoded map. Fallback: non_bank."""
    return COMPANY_TYPE_MAP.get(ticker.upper(),
           COMPANY_TYPE_MAP.get(ticker, 'non_bank'))
 
 
def get_unit(ticker: str, pdf_text_first5: str = '') -> str:
    """
    Determine unit for this ticker.
    Hardcoded overrides take priority over PDF text detection.
    For banks: scan PDF text to confirm kDT (they always mark it).
    """
    if ticker in ALWAYS_DT_TICKERS:
        return 'DT'
 
    # For banks and unknown tickers: scan PDF text
    kdt_pats = [
        r'en\s+1[\s.]000\s+DT', r'millier', r'kDT',
        r'1\.000\s+dinars', r'milliers\s+de\s+dinars',
    ]
    for p in kdt_pats:
        if re.search(p, pdf_text_first5, re.IGNORECASE):
            return 'kDT'
 
    dt_pats = [
        r'en\s+[Dd]inar\s+[Tt]unisien',
        r'exprim[ée]\s+en\s+[Dd]inars?\b',
    ]
    for p in dt_pats:
        if re.search(p, pdf_text_first5, re.IGNORECASE):
            return 'DT'
 
    # Banks default to kDT, everything else defaults to DT
    ctype = get_company_type(ticker)
    return 'kDT' if ctype == 'bank' else 'DT'
 
 
print("Cell 2 OK — company maps defined")
print(f"  {len(COMPANY_TYPE_MAP)} tickers mapped")
print(f"  {len(ALWAYS_DT_TICKERS)} tickers forced to DT unit")
 
 


Cell 2 OK — company maps defined
  66 tickers mapped
  34 tickers forced to DT unit


In [3]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 3 — PDF page finder and text extractor             ║
# ╚══════════════════════════════════════════════════════════╝
#
# WHAT CHANGED:
#   - find_key_pages now uses COMPANY_TYPE_MAP instead of guessing
#   - extract_pages_text now returns MORE text (10,000 chars)
#     because GPT-4o has large context and accuracy matters more
#     than token savings with the enhanced prompts
#   - Added get_first5_text() for unit detection
 
BILAN_KW = [
    'Total des actifs', 'Total actifs', 'TOTAL ACTIF',
    'Caisse et avoirs', 'AC1', 'AC 1', 'ACTIFS',
    'TOTAL BILAN', 'Total général du bilan',
]
PNL_KW = [
    'Produit Net Bancaire', 'Produit net Bancaire',
    'Etat de Résultat', 'ETAT DE RESULTAT', 'ETAT DES RESULTATS',
    "Chiffre d'affaires", 'COMPTE DE RESULTAT',
    'Résultat net', "Produits d'exploitation",
    'COMPTE DE PROFITS ET PERTES',
]
PRUD_KW = [
    'LCR', 'Taux des engagements', 'Taux de couverture',
    'créances classées', 'Ratio de liquidité',
]
 
 
def get_first5_text(pdf_path: Path) -> str:
    """Extract raw text from first 5 pages for unit detection."""
    text = ''
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages[:5]:
                text += (page.extract_text() or '')
    except:
        pass
    return text
 
 
def find_key_pages(pdf_path: Path, ticker: str = '') -> dict:
    """
    Scan PDF to find which pages contain balance sheet, P&L,
    and prudential ratios. Uses company type map for accuracy.
    """
    result = {
        'bilan': None, 'pnl': None, 'prudential': None,
        'bilan_pages': [], 'total_pages': 0,
    }
    try:
        with pdfplumber.open(pdf_path) as pdf:
            result['total_pages'] = len(pdf.pages)
 
            for i, page in enumerate(pdf.pages):
                text    = page.extract_text() or ''
                text_up = text.upper()
 
                # Balance sheet detection
                if result['bilan'] is None:
                    has_bilan = 'BILAN' in text_up
                    has_actif = any(kw.upper() in text_up for kw in BILAN_KW)
                    has_nums  = bool(re.search(r'\d{3,}[\s\d]{4,}', text))
                    if (has_bilan or has_actif) and has_nums:
                        result['bilan'] = i
                        result['bilan_pages'].append(i)
 
                # Balance sheet continuation (split across 2 pages)
                elif (i == result['bilan'] + 1 and result['pnl'] is None):
                    has_passif = ('PASSIF' in text_up or
                                  'CAPITAUX PROPRES' in text_up)
                    is_not_pnl = sum(1 for kw in PNL_KW if kw in text) == 0
                    if has_passif and is_not_pnl:
                        result['bilan_pages'].append(i)
 
                # P&L detection
                if result['pnl'] is None:
                    pnl_score = sum(1 for kw in PNL_KW if kw in text)
                    if pnl_score >= 1 and i not in result['bilan_pages']:
                        result['pnl'] = i
 
                # Prudential ratios (banks only)
                if result['prudential'] is None:
                    prud_score = sum(1 for kw in PRUD_KW if kw in text)
                    if prud_score >= 2:
                        result['prudential'] = i
 
                # Stop early if all found
                if (result['bilan'] is not None and
                        result['pnl'] is not None and
                        result['prudential'] is not None):
                    break
 
    except Exception as e:
        result['error'] = str(e)
 
    return result
 
 
def extract_pages_text(pdf_path: Path,
                       page_indices: list,
                       max_chars: int = 10000) -> str:
    """
    Extract and concatenate text from specific PDF pages.
    10,000 chars — more context = better extraction accuracy.
    GPT-4o handles this comfortably within its context window.
    """
    all_text = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for idx in sorted(set(page_indices)):
                if 0 <= idx < len(pdf.pages):
                    raw     = pdf.pages[idx].extract_text() or ''
                    # Clean excessive whitespace
                    cleaned = re.sub(r'[ \t]+', ' ', raw)
                    cleaned = re.sub(r'\n{3,}', '\n\n', cleaned)
                    cleaned = cleaned.strip()
                    if cleaned:
                        all_text.append(f"[PAGE {idx+1}]\n{cleaned}")
    except Exception as e:
        print(f"  Text extraction error: {e}")
        return ''
 
    combined = '\n\n'.join(all_text)
    if len(combined) > max_chars:
        combined = combined[:max_chars] + '\n[TRUNCATED]'
    return combined
 
 
print("Cell 3 OK — page finder and text extractor defined")
 
 

Cell 3 OK — page finder and text extractor defined


In [4]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 4 — Enhanced extraction prompts                    ║
# ╚══════════════════════════════════════════════════════════╝
#
# THREE PROMPTS — one per company category:
#
# PROMPT_BANK: for banks (AMEN BANK, ATB, BIAT, BNA, STB etc.)
#   - Values in kDT — return as-is
#   - Expects BCT row codes AC1-AC7, PA1-PA5
#   - Extracts PNB (not revenue)
#   - Extracts NPL ratio, coverage ratio, LCR from notes
#
# PROMPT_KDT_NONBANK: for non-banks that use kDT (rare)
#   - Returns revenue (not PNB)
#   - Explicitly told to return null for bank-only fields
#
# PROMPT_DT: for leasing, insurance, most non-banks using full DT
#   - MUST divide all monetary values by 1000
#   - Includes worked examples with actual Tunisian numbers
#   - Returns revenue (not PNB)
#   - Extra warnings about common mistakes
#
# WHY THREE INSTEAD OF TWO:
#   The old two-prompt system confused leasing companies (which
#   have "Produit Net" in their P&L) with banks. Separate prompt
#   for banks removes all ambiguity.
 
SYSTEM_PROMPT = """You are a precise financial data extraction assistant.
You specialize in Tunisian company annual reports (états financiers).
You extract specific numbers from PDF text and return valid JSON only.
You never add explanations, markdown, code fences, or commentary.
You return only the raw JSON object, nothing else."""
 
 
PROMPT_BANK = """You are extracting from a TUNISIAN BANK annual report.
 
UNIT: kDT (thousands of Tunisian Dinars). Return all values AS-IS — do NOT divide.
 
CRITICAL RULES:
1. Return ONLY a valid JSON object — absolutely no markdown, no text, no ```
2. Take values from the MOST RECENT column (leftmost, or labeled with current year)
3. Use null (not 0, not "N/A") for any field you cannot find with confidence
4. All monetary values are already in kDT — return the number exactly as shown
5. NEVER return a number where net_result > total_assets — that means you picked wrong row
6. Negative values: use -4401 not (4401) and not "-4 401"
7. Remove all spaces from numbers: 12 348 844 → 12348844
 
FROM THE BALANCE SHEET (titled "Bilan" or "Bilan au 31/12/YYYY"):
Look for BCT standard rows:
- total_assets: the final "Total des actifs" or "TOTAL ACTIF" line — the grand total
  (NOT subtotals like "Total des actifs courants" or individual asset lines)
- total_liabilities: "Total des passifs" (NOT "Total passifs et capitaux propres")
- total_loans: row labeled AC3 or "Créances sur la clientèle"
- total_deposits: row labeled PA3 or "Dépôts et avoirs de la clientèle"
- equity: "Total des capitaux propres" — must be POSITIVE for a healthy bank
  (typically 500,000 to 3,000,000 for Tunisian banks)
 
FROM THE P&L (titled "Etat de Résultat" or "Etat des résultats"):
- pnb: "Produit Net Bancaire" — a key bank metric (typically 200,000 to 800,000)
- revenue: null — banks use pnb, not revenue
- net_result: "Résultat net de l'exercice" or "Résultat net" (typically 50,000-300,000)
- operating_expenses: "Charges opératoires" or "Frais généraux" (return as positive number)
 
FROM NOTES OR APPENDIX (may be on later pages):
- npl_ratio: "Taux des engagements classés" — a percentage e.g. 9.96 (not 996 or 0.0996)
- coverage_ratio: "Taux de couverture des créances classées" — e.g. 75.18
- lcr: "LCR" or "Ratio de liquidité à court terme" — e.g. 142.97
 
SANITY CHECK before returning:
- total_assets should be between 3,000,000 and 25,000,000 for Tunisian banks
- equity should be between 300,000 and 3,000,000
- net_result should be between 30,000 and 400,000
- pnb should be between 150,000 and 1,000,000
If any value falls far outside these ranges, you probably picked the wrong row — use null instead.
 
PDF TEXT:
{text}
 
Return exactly this JSON structure:
{{"total_assets":null,"total_liabilities":null,"total_loans":null,"total_deposits":null,"equity":null,"pnb":null,"revenue":null,"net_result":null,"operating_expenses":null,"npl_ratio":null,"coverage_ratio":null,"lcr":null}}"""
 
 
PROMPT_DT = """You are extracting from a Tunisian NON-BANK company annual report.
This company is a {company_type} company.
 
UNIT: FULL DINARS (DT). You MUST divide every monetary value by 1000.
This is mandatory — the PDF shows full dinars, the database stores kDT (thousands).
 
CONVERSION EXAMPLES (memorize these):
  PDF shows: 272 682 126  →  you return: 272682.126
  PDF shows: 40 416 678   →  you return: 40416.678
  PDF shows: 8 734 291    →  you return: 8734.291
  PDF shows: 893 495 750  →  you return: 893495.750
  PDF shows: -4 599 351   →  you return: -4599.351
 
DO NOT DIVIDE: npl_ratio, coverage_ratio, lcr — these are percentages, already correct.
 
CRITICAL RULES:
1. Return ONLY a valid JSON object — no markdown, no text, no ```
2. Divide ALL monetary values by 1000 before returning
3. Use null for fields you cannot find — never estimate
4. Take values from the MOST RECENT column (leftmost or current year)
5. NEVER return net_result larger than total_assets — you divided wrong
6. Remove spaces from numbers before dividing: "8 734 291" → 8734291 → 8734.291
 
FROM BALANCE SHEET ("Bilan" or "Bilan au 31/12/YYYY"):
- total_assets: grand total line "Total des actifs" or "Total général du bilan" or "TOTAL BILAN"
  WARNING: skip subtotals — "Total des actifs immobilisés", "Total des actifs courants"
  The grand total is always the largest single number on the balance sheet.
- total_liabilities: "Total des passifs" or "Total des dettes" — grand total only
- total_loans: null (this is a non-bank — no loan portfolio)
- total_deposits: null (this is a non-bank — no deposit base)
- equity: "Total des capitaux propres" or "Capitaux propres" total
  (NOT "Total des capitaux propres et des passifs" — that equals total_assets)
 
FROM P&L ("Compte de Résultat" or "Etat de Résultat" or "Etat des résultats"):
- pnb: null — non-banks do not have Produit Net Bancaire
- revenue: "Chiffre d'affaires" or "Revenus des activités ordinaires" or "Total des produits"
  For leasing companies: look for "Produits des loyers" or "Intérêts et revenus assimilés"
  For insurance companies: look for "Primes émises" or "Chiffre d'affaires"
- net_result: "Résultat net de l'exercice" or "Résultat de l'exercice" or "Bénéfice net"
- operating_expenses: "Charges d'exploitation" or "Total des charges" (return as positive)
 
NOTES:
- npl_ratio: null (non-banks do not report this)
- coverage_ratio: null (non-banks do not report this)
- lcr: null (non-banks do not report this)
 
SANITY CHECK after dividing by 1000:
- total_assets for a mid-size Tunisian company: 10,000 to 2,000,000 kDT
- net_result should be less than 30% of total_assets
- equity should be positive and less than total_assets
If values seem wrong after dividing, you may have divided twice — check your work.
 
PDF TEXT:
{text}
 
Return exactly this JSON structure:
{{"total_assets":null,"total_liabilities":null,"total_loans":null,"total_deposits":null,"equity":null,"pnb":null,"revenue":null,"net_result":null,"operating_expenses":null,"npl_ratio":null,"coverage_ratio":null,"lcr":null}}"""
 
 
PROMPT_RETRY = """You previously extracted financial data from this PDF but some values
appear incorrect. Here is what was returned and why it seems wrong:
 
PREVIOUS RESULT: {previous_result}
PROBLEM DETECTED: {problem_description}
 
Please re-read the PDF text carefully and fix the errors.
Pay special attention to:
1. Are you reading the correct column (most recent year)?
2. Did you divide by 1000 if the unit is DT? (required for {company_type})
3. Did you accidentally pick a subtotal instead of the grand total?
4. Is net_result larger than total_assets? If yes, you have a unit error.
 
Company type: {company_type}
Expected unit after conversion: kDT
 
PDF TEXT:
{text}
 
Return only the corrected JSON — no explanation:
{{"total_assets":null,"total_liabilities":null,"total_loans":null,"total_deposits":null,"equity":null,"pnb":null,"revenue":null,"net_result":null,"operating_expenses":null,"npl_ratio":null,"coverage_ratio":null,"lcr":null}}"""
 
 
print("Cell 4 OK — 3 prompts defined (bank / DT non-bank / retry)")
 
 

Cell 4 OK — 3 prompts defined (bank / DT non-bank / retry)


In [5]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 5 — GPT-4o API call with token rotation            ║
# ╚══════════════════════════════════════════════════════════╝
#
# HOW TWO-PASS EXTRACTION WORKS:
#   Pass 1: Normal extraction with appropriate prompt
#   Validation check: is net_result > 30% of total_assets?
#   If yes → Pass 2: retry with PROMPT_RETRY explaining the error
#   Pass 2 result overwrites Pass 1 only if it's better
#
# HOW TOKEN ROTATION WORKS:
#   _token_index tracks which token is active
#   After REQUESTS_PER_TOKEN requests, moves to next token
#   If 429 error (rate limit), also rotates immediately
#   If all tokens exhausted, returns 'all_tokens_exhausted'
 
def _rotate_token() -> bool:
    """
    Switch to next available token.
    Returns True if rotation succeeded, False if all exhausted.
    """
    global _token_index
    next_idx = _token_index + 1
    if next_idx >= len(GITHUB_TOKENS):
        return False  # All tokens exhausted
    _token_index = next_idx
    print(f"\n  [Token rotated → account {_token_index + 1}/{len(GITHUB_TOKENS)} "
          f"| {_token_requests[_token_index]} requests used today]",
          end='', flush=True)
    return True
 
 
def _call_api(prompt: str, retries: int = 2) -> tuple:
    """
    Make one API call with retry logic.
    Returns (response_text, error_string).
    error_string is None on success.
    """
    global _token_requests
 
    for attempt in range(1, retries + 1):
        # Check if current token is exhausted before calling
        if _token_requests[_token_index] >= REQUESTS_PER_TOKEN:
            if not _rotate_token():
                return None, 'all_tokens_exhausted'
 
        try:
            client = get_client()
            response = client.chat.completions.create(
                model=GPT_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": prompt}
                ],
                max_tokens=400,
                temperature=0.0,  # deterministic — no creativity for numbers
            )
            _token_requests[_token_index] += 1
            return response.choices[0].message.content.strip(), None
 
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate' in err.lower() or 'RateLimit' in err:
                # Rate limit — try rotating token
                if not _rotate_token():
                    return None, 'all_tokens_exhausted'
                time.sleep(3)
            elif '500' in err or 'server' in err.lower():
                if attempt < retries:
                    time.sleep(15)
                else:
                    return None, f'server_error:{err[:60]}'
            else:
                if attempt < retries:
                    time.sleep(8)
                else:
                    return None, f'api_error:{err[:80]}'
 
    return None, 'max_retries_exceeded'
 
 
def _parse_json(raw: str) -> dict:
    """Strip markdown fences and parse JSON."""
    raw = re.sub(r'^```(?:json)?\s*', '', raw.strip())
    raw = re.sub(r'\s*```$', '', raw)
    return json.loads(raw.strip())
 
 
def call_gpt(text: str,
             ticker: str,
             company_type: str,
             unit: str) -> tuple:
    """
    Two-pass extraction:
    Pass 1 — extract with correct prompt for company type/unit
    Pass 2 — retry if suspicious values detected (unit errors)
 
    Returns (data_dict, notes_string, used_retry_bool)
    """
    if not text or not text.strip():
        return {}, 'no_text_extracted', False
 
    # ── Select prompt for Pass 1 ──────────────────────────
    if company_type == 'bank':
        prompt1 = PROMPT_BANK.format(text=text)
    else:
        prompt1 = PROMPT_DT.format(
            text=text, company_type=company_type)
 
    # ── Pass 1 ────────────────────────────────────────────
    raw1, err1 = _call_api(prompt1)
    if err1:
        return {}, err1, False
 
    try:
        data1 = _parse_json(raw1)
    except Exception as e:
        return {}, f'json_parse_error:{str(e)[:60]}', False
 
    # ── Check if Pass 1 has suspicious values ─────────────
    ta = data1.get('total_assets')
    nr = data1.get('net_result')
    eq = data1.get('equity')
 
    problem = None
    if ta and nr and abs(nr) > abs(ta) * 0.50:
        problem = (f"net_result ({nr}) is more than 30% of "
                   f"total_assets ({ta}) — likely unit error")
    elif ta and ta > 0 and eq and eq < 0 and company_type == 'bank':
        problem = (f"equity ({eq}) is negative for a bank — "
                   f"likely wrong row selected")
    elif ta and ta < 500 and company_type in ('bank', 'leasing'):
        problem = (f"total_assets ({ta} kDT) is unrealistically "
                   f"small for a {company_type}")
 
    if not problem:
        return data1, 'OK', False
 
    # ── Pass 2 — retry with problem description ────────────
    print(f"\n    [Pass 2 retry: {problem[:60]}]", end='', flush=True)
 
    prompt2 = PROMPT_RETRY.format(
        previous_result=json.dumps(data1),
        problem_description=problem,
        company_type=company_type,
        text=text
    )
 
    raw2, err2 = _call_api(prompt2)
    if err2:
        # Pass 2 failed — return Pass 1 result with warning
        return data1, f'retry_failed:{err2}', True
 
    try:
        data2 = _parse_json(raw2)
    except:
        return data1, f'retry_json_error', True
 
    # Use Pass 2 result only if it's actually better
    ta2 = data2.get('total_assets')
    nr2 = data2.get('net_result')
    if ta2 and nr2 and abs(nr2) <= abs(ta2) * 0.30:
        return data2, f'retry_fixed:{problem[:50]}', True
    else:
        # Pass 2 still wrong — return whichever has better assets
        if ta2 and ta and abs(ta2) > abs(ta):
            return data2, f'retry_used_anyway', True
        return data1, f'retry_no_improvement', True
 
 
print("Cell 5 OK — call_gpt with two-pass extraction defined")
print(f"Requests used today: {_token_requests}")
 
 


Cell 5 OK — call_gpt with two-pass extraction defined
Requests used today: [1, 0, 0, 0]


In [6]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 6 — Validation                                     ║
# ╚══════════════════════════════════════════════════════════╝
#
# IMPROVEMENTS vs previous version:
#   - Confidence HARD CAPPED at 1.0 (fixes 1.143 bug)
#   - pnb in non-bank record → flagged as needs_review
#   - equity < 0 for bank → flagged as needs_review
#   - Separate field counts for banks vs non-banks
#     (banks have 8 meaningful fields, non-banks have 5)
#     This gives fairer confidence scores
 
BANK_MAIN_FIELDS    = ['total_assets', 'total_liabilities',
                       'total_loans', 'total_deposits',
                       'equity', 'pnb', 'net_result', 'operating_expenses']
NONBANK_MAIN_FIELDS = ['total_assets', 'total_liabilities',
                       'equity', 'revenue', 'net_result']
RATIO_FIELDS        = ['npl_ratio', 'coverage_ratio', 'lcr']
 
 
def validate_extraction(data: dict, company_type: str) -> tuple:
    """
    Validate and clean extracted values.
    Returns (cleaned_data, confidence, needs_review, notes)
    """
    notes        = []
    needs_review = False
    valid_count  = 0
 
    # Range limits per company type
    if company_type == 'bank':
        max_kdt      = 30_000_000
        min_kdt      = 100
        main_fields  = BANK_MAIN_FIELDS
    elif company_type in ('leasing', 'insurance'):
        max_kdt      = 5_000_000
        min_kdt      = 10
        main_fields  = NONBANK_MAIN_FIELDS
    else:
        max_kdt      = 10_000_000
        min_kdt      = 1
        main_fields  = NONBANK_MAIN_FIELDS
 
    # Validate monetary fields
    for field in main_fields + ['operating_expenses']:
        val = data.get(field)
        if val is None:
            continue
 
        if not isinstance(val, (int, float)):
            data[field] = None
            notes.append(f'{field}:not_numeric')
            continue
 
        if abs(val) > 0:
            if abs(val) < min_kdt:
                data[field] = None
                notes.append(f'{field}:too_small({val:.1f})')
                continue
 
            if abs(val) > max_kdt:
                # Try auto-correction: divide by 1000
                corrected = val / 1000
                if min_kdt <= abs(corrected) <= max_kdt:
                    data[field] = round(corrected, 3)
                    notes.append(
                        f'{field}:auto_div1000({val:.0f}→{corrected:.0f})')
                    val = corrected
                else:
                    data[field] = None
                    notes.append(f'{field}:out_of_range({val:.0f})')
                    continue
 
        if field in main_fields and data.get(field) is not None:
            valid_count += 1
 
    # Validate ratio fields (must be 0–300%)
    for field in RATIO_FIELDS:
        val = data.get(field)
        if val is None:
            continue
        if not isinstance(val, (int, float)) or not (0 < val < 300):
            data[field] = None
            notes.append(f'{field}:invalid({val})')
 
    # Cross-checks
    ta = data.get('total_assets')
    tl = data.get('total_liabilities')
    eq = data.get('equity')
    nr = data.get('net_result')
 
    # Net result > 30% of assets → unit error still present
    if ta and nr and abs(nr) > abs(ta) * 0.50:
        needs_review = True
        notes.append(f'net_gt_30pct_assets({nr:.0f}>{ta:.0f})')
 
    # Balance sheet check
    if ta and tl and eq and ta > 0:
        diff_pct  = abs(ta - (tl + eq)) / ta
        tolerance = 0.05 if company_type == 'bank' else 0.30
        if diff_pct > tolerance:
            needs_review = True
            notes.append(f'balance_diff:{diff_pct*100:.1f}%')
 
    # Missing total_assets always flags review
    if not data.get('total_assets'):
        needs_review = True
        notes.append('total_assets:missing')
 
    # pnb in non-bank record
    if company_type != 'bank' and data.get('pnb'):
        data['pnb'] = None
        notes.append('pnb:cleared_nonbank')
 
    # Confidence: fraction of expected fields found, HARD CAPPED at 1.0
    confidence = round(min(valid_count / len(main_fields), 1.0), 3)
    note_str   = ' | '.join(notes) if notes else 'OK'
 
    return data, confidence, needs_review, note_str
 
 
print("Cell 6 OK — validate_extraction defined")
 
 


Cell 6 OK — validate_extraction defined


In [7]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 7 — Database helpers                               ║
# ╚══════════════════════════════════════════════════════════╝
 
def get_isin_map() -> dict:
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute('SELECT ticker, isin_code FROM company_metadata')
            return {r[0]: r[1] for r in cur.fetchall()}
    finally:
        conn.close()
 
 
def record_already_good(ticker: str, period: str,
                        min_conf: float = 0.5) -> bool:
    """Returns True if this record already has conf >= min_conf."""
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute('''
                SELECT extraction_confidence, needs_review
                FROM financial_statements
                WHERE ticker = %s AND period = %s
            ''', (ticker, period))
            row = cur.fetchone()
            if not row or not row[0]:
                return False
            conf = float(row[0])
            review = row[1]
            # Skip only if good confidence AND not flagged for review
            return conf >= min_conf and not review
    finally:
        conn.close()
 
 
def compute_derived(data: dict, period: str) -> dict:
    """Compute ROE, cost-income ratio, loan-to-deposit from raw values."""
    net = data.get('net_result')
    eq  = data.get('equity')
    if net and eq and eq > 0:
        # FY annual: no annualisation needed
        data['roe'] = round((net / eq) * 100, 4)
 
    pnl_base = data.get('pnb') or data.get('revenue')
    opex     = data.get('operating_expenses')
    if opex and pnl_base and pnl_base > 0:
        data['cost_income_ratio'] = round(abs(opex) / pnl_base * 100, 4)
 
    loans = data.get('total_loans')
    dep   = data.get('total_deposits')
    if loans and dep and dep > 0:
        data['loan_to_deposit'] = round(loans / dep * 100, 4)
 
    return data
 
 
def upsert_record(ticker, isin, period, period_end_date,
                  source_pdf, company_type, data,
                  confidence, needs_review, notes) -> bool:
    data = compute_derived(data.copy(), period)
    conn = get_conn()
    try:
        with conn.cursor() as cur:
            cur.execute('''
                INSERT INTO financial_statements (
                    ticker, isin_code, period, period_end_date, source_pdf,
                    company_type,
                    total_assets, total_liabilities, total_loans,
                    total_deposits, equity,
                    pnb, revenue, net_result, operating_expenses,
                    npl_ratio, coverage_ratio, lcr,
                    roe, cost_income_ratio, loan_to_deposit,
                    extraction_confidence, needs_review, extraction_notes
                ) VALUES (
                    %s,%s,%s,%s,%s,%s,
                    %s,%s,%s,%s,%s,
                    %s,%s,%s,%s,
                    %s,%s,%s,
                    %s,%s,%s,
                    %s,%s,%s
                )
                ON CONFLICT (ticker, period) DO UPDATE SET
                    total_assets          = EXCLUDED.total_assets,
                    total_liabilities     = EXCLUDED.total_liabilities,
                    total_loans           = EXCLUDED.total_loans,
                    total_deposits        = EXCLUDED.total_deposits,
                    equity                = EXCLUDED.equity,
                    pnb                   = EXCLUDED.pnb,
                    revenue               = EXCLUDED.revenue,
                    net_result            = EXCLUDED.net_result,
                    operating_expenses    = EXCLUDED.operating_expenses,
                    npl_ratio             = EXCLUDED.npl_ratio,
                    coverage_ratio        = EXCLUDED.coverage_ratio,
                    lcr                   = EXCLUDED.lcr,
                    roe                   = EXCLUDED.roe,
                    cost_income_ratio     = EXCLUDED.cost_income_ratio,
                    loan_to_deposit       = EXCLUDED.loan_to_deposit,
                    extraction_confidence = EXCLUDED.extraction_confidence,
                    needs_review          = EXCLUDED.needs_review,
                    extraction_notes      = EXCLUDED.extraction_notes,
                    scraped_at            = NOW()
            ''', (
                ticker, isin, period, period_end_date,
                source_pdf, company_type,
                data.get('total_assets'), data.get('total_liabilities'),
                data.get('total_loans'), data.get('total_deposits'),
                data.get('equity'),
                data.get('pnb'), data.get('revenue'),
                data.get('net_result'), data.get('operating_expenses'),
                data.get('npl_ratio'), data.get('coverage_ratio'),
                data.get('lcr'),
                data.get('roe'), data.get('cost_income_ratio'),
                data.get('loan_to_deposit'),
                confidence, needs_review, notes
            ))
            new_row = cur.rowcount == 1
        conn.commit()
        return new_row
    except Exception as e:
        conn.rollback()
        print(f"\n  DB error {ticker} {period}: {e}")
        return False
    finally:
        conn.close()
 
 
print("Cell 7 OK — DB helpers defined")
 
 


Cell 7 OK — DB helpers defined


In [8]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 8 — Master extractor for one PDF                   ║
# ╚══════════════════════════════════════════════════════════╝
 
def extract_one_pdf(pdf_path: Path,
                    ticker: str,
                    period: str,
                    period_end_date: str,
                    isin_map: dict,
                    force: bool = False) -> dict:
    """
    Full extraction pipeline for one FY_ANNUAL PDF.
 
    Steps:
    1. Skip if already extracted with good confidence
    2. Get company type from hardcoded map
    3. Get unit from hardcoded map + PDF text confirmation
    4. Find key pages (balance sheet, P&L, prudential)
    5. Extract text from those pages + baseline pages
    6. Call GPT-4o with correct prompt (two-pass if needed)
    7. Validate and clean the JSON response
    8. Insert into database
    """
    status = {
        'ticker':     ticker,
        'period':     period,
        'filename':   pdf_path.name,
        'confidence': 0.0,
        'review':     False,
        'notes':      '',
        'skipped':    False,
        'used_retry': False,
    }
 
    # Step 1 — Skip if already good
    if not force and record_already_good(ticker, period):
        status['skipped'] = True
        status['notes']   = 'already_extracted'
        return status
 
    # Step 2 — Company type from hardcoded map
    company_type = get_company_type(ticker)
 
    # Step 3 — Unit detection
    first5_text = get_first5_text(pdf_path)
    unit        = get_unit(ticker, first5_text)
 
    # Step 4 — Find key pages
    kp          = find_key_pages(pdf_path, ticker)
    total_pages = kp.get('total_pages', 0)
 
    detected = list(kp.get('bilan_pages', []))
    for key in ('pnl', 'prudential'):
        idx = kp.get(key)
        if idx is not None:
            detected.append(idx)
 
    # Always include first BASELINE_PAGES as safety net
    baseline      = list(range(min(BASELINE_PAGES, total_pages)))
    pages_to_send = sorted(set(detected + baseline))
 
    # Step 5 — Extract text
    text = extract_pages_text(pdf_path, pages_to_send, max_chars=10000)
 
    if not text:
        upsert_record(ticker, isin_map.get(ticker), period,
                      period_end_date, pdf_path.name, company_type,
                      {}, 0.0, True, 'text_extraction_failed')
        status['notes']  = 'text_extraction_failed'
        status['review'] = True
        return status
 
    # Step 6 — Call GPT-4o
    data, api_note, used_retry = call_gpt(text, ticker, company_type, unit)
    status['used_retry'] = used_retry
 
    # Stop batch if all tokens exhausted
    if api_note == 'all_tokens_exhausted':
        status['notes']   = 'all_tokens_exhausted'
        status['skipped'] = True
        return status
 
    if not data:
        upsert_record(ticker, isin_map.get(ticker), period,
                      period_end_date, pdf_path.name, company_type,
                      {}, 0.0, True, f'gpt_failed:{api_note[:60]}')
        status['notes']  = f'gpt_failed:{api_note[:60]}'
        status['review'] = True
        return status
 
    # Step 7 — Validate
    data, confidence, needs_review, val_notes = validate_extraction(
        data, company_type)
 
    final_notes = val_notes
    if api_note != 'OK':
        final_notes = f'{api_note} | {val_notes}'
 
    # Step 8 — Insert
    upsert_record(
        ticker, isin_map.get(ticker), period, period_end_date,
        pdf_path.name, company_type, data,
        confidence, needs_review, final_notes
    )
 
    status['confidence'] = confidence
    status['review']     = needs_review
    status['notes']      = final_notes
    return status
 
 
print("Cell 8 OK — extract_one_pdf defined")
 
 


Cell 8 OK — extract_one_pdf defined


In [11]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 9 — Clean slate + test on 4 PDFs                  ║
# ╚══════════════════════════════════════════════════════════╝
#
# PART A: Delete all existing rows from financial_statements
# PART B: Test on 4 known PDFs (1 bank, 1 leasing, 1 non-bank, 1 insurance)
#
# Uses 4 API requests from your daily quota.
# ONLY run Part A once — it deletes everything.
# After running, check results before proceeding to Cell 10.
 
isin_map = get_isin_map()
print(f"ISIN map loaded: {len(isin_map)} stocks")
print()
 
# ── PART A: Clean slate ───────────────────────────────────
print("PART A — Deleting all existing financial_statements rows...")
conn = get_conn()
try:
    with conn.cursor() as cur:
        cur.execute('SELECT COUNT(*) FROM financial_statements')
        before = cur.fetchone()[0]
        cur.execute('DELETE FROM financial_statements')
        conn.commit()
    print(f"  Deleted {before} rows. Table is now empty.")
except Exception as e:
    conn.rollback()
    print(f"  Error: {e}")
finally:
    conn.close()
 
print()
 
# ── PART B: Test on 4 PDFs ────────────────────────────────
TEST_CASES = [
    # (path, ticker, period, period_end_date, expected_values)
    (
        PDF_BASE_DIR / 'AMEN BANK' / 'FY_ANNUAL' / 'amen_bank_efd311224.pdf',
        'AMEN BANK', 'FY 2024', '2024-12-31',
        {'total_assets': 11855711, 'pnb': 566487, 'net_result': 229957}
    ),
    (
        PDF_BASE_DIR / 'ATL' / 'FY_ANNUAL' / 'atl_efd311224.pdf',
        'ATL', 'FY 2024', '2024-12-31',
        {}  # leasing — just check conf > 0
    ),
    (
        PDF_BASE_DIR / 'ARTES' / 'FY_ANNUAL' / 'artes_efd311224.pdf',
        'ARTES', 'FY 2024', '2024-12-31',
        {'revenue': 257332}
    ),
    (
        PDF_BASE_DIR / 'ASSUR MAGHREBIA' / 'FY_ANNUAL' / 'assurances_mag_efd311224.pdf',
        'ASSUR MAGHREBIA', 'FY 2024', '2024-12-31',
        {}  # insurance — just check conf > 0
    ),
]
 
print("Running 4 format tests (uses 4-8 API requests)...")
print()
 
all_passed = True
tests_run  = 0
 
for pdf_path, ticker, period, ped, expected in TEST_CASES:
    if not pdf_path.exists():
        # Try to find any FY_ANNUAL PDF for this ticker
        alt_dir = PDF_BASE_DIR / ticker / 'FY_ANNUAL'
        if alt_dir.exists():
            pdfs = list(alt_dir.glob('*.pdf'))
            if pdfs:
                pdf_path = pdfs[-1]  # use most recent
                print(f"  [{ticker}] using {pdf_path.name}")
            else:
                print(f"  SKIP {ticker} — no FY_ANNUAL PDFs found")
                print()
                continue
        else:
            print(f"  SKIP {ticker} — folder not found: {alt_dir}")
            print()
            continue
 
    tests_run += 1
    ctype = get_company_type(ticker)
    unit  = get_unit(ticker, get_first5_text(pdf_path))
 
    print(f"  [{ticker} {period}]")
    print(f"    file:         {pdf_path.name}")
    print(f"    company_type: {ctype}")
    print(f"    unit:         {unit}")
    print(f"    requests used before: "
          f"{sum(_token_requests)}/{len(GITHUB_TOKENS)*REQUESTS_PER_TOKEN}")
 
    status = extract_one_pdf(pdf_path, ticker, period, ped, isin_map, force=True)
 
    icon = '✓' if not status['review'] else '⚠'
    retry_flag = ' [2-pass]' if status['used_retry'] else ''
    print(f"    {icon} conf={status['confidence']:.2f}  "
          f"{status['notes'][:70]}{retry_flag}")
 
    # Check DB values
    if expected:
        conn = get_conn()
        with conn.cursor() as cur:
            cur.execute('''
                SELECT total_assets, equity, net_result, pnb, revenue,
                       extraction_confidence, needs_review
                FROM financial_statements
                WHERE ticker=%s AND period=%s
            ''', (ticker, period))
            row = cur.fetchone()
        conn.close()
 
        if row:
            print(f"    DB: assets={row[0]}  equity={row[1]}  "
                  f"net={row[2]}  pnb={row[3]}  rev={row[4]}")
            field_map = {'total_assets': 0, 'equity': 1,
                         'net_result': 2, 'pnb': 3, 'revenue': 4}
            for field, exp_val in expected.items():
                got = row[field_map[field]]
                if got and abs(float(got) - exp_val) / max(exp_val, 1) < 0.05:
                    print(f"    ✓ {field}: {got} (~{exp_val})")
                else:
                    print(f"    ✗ {field}: {got} (expected ~{exp_val})")
                    all_passed = False
 
    print(f"    requests used after: "
          f"{sum(_token_requests)}/{len(GITHUB_TOKENS)*REQUESTS_PER_TOKEN}")
    print()
    time.sleep(3)
 
print()
total_used  = sum(_token_requests)
total_limit = len(GITHUB_TOKENS) * REQUESTS_PER_TOKEN
if tests_run == 0:
    print("No test PDFs found — check PDF_BASE_DIR")
elif all_passed:
    print(f"ALL TESTS PASSED ✓")
    print(f"Requests used: {total_used}/{total_limit}")
    print(f"Remaining today: {total_limit - total_used}")
    print("→ Proceed to Cell 10")
else:
    print("SOME TESTS FAILED ⚠ — review output above before batch")
 
 

ISIN map loaded: 70 stocks

PART A — Deleting all existing financial_statements rows...
  Deleted 369 rows. Table is now empty.

Running 4 format tests (uses 4-8 API requests)...

  [AMEN BANK FY 2024]
    file:         amen_bank_efd311224.pdf
    company_type: bank
    unit:         kDT
    requests used before: 1/188
    ✓ conf=1.00  OK
    DB: assets=11855711.000  equity=1574007.000  net=229957.000  pnb=566487.000  rev=None
    ✓ total_assets: 11855711.000 (~11855711)
    ✓ pnb: 566487.000 (~566487)
    ✓ net_result: 229957.000 (~229957)
    requests used after: 2/188

  [ATL FY 2024]
    file:         atl_efd311224.pdf
    company_type: leasing
    unit:         DT
    requests used before: 2/188
    ✓ conf=1.00  OK
    requests used after: 3/188

  [ARTES FY 2024]
    file:         artes_efd311224.pdf
    company_type: non_bank
    unit:         DT
    requests used before: 3/188
    ✓ conf=1.00  OK
    DB: assets=288276.127  equity=140945.761  net=40416.678  pnb=None  rev=257331.

In [11]:
# ── Reset token counters for today's fresh run ──────────
# Run this cell at the start of each day BEFORE Cell 10
# to clear any requests used during testing
_token_requests = [0] * len(GITHUB_TOKENS)
_token_index    = 0
print(f"Token counters reset.")
print(f"Fresh capacity today: {len(GITHUB_TOKENS) * REQUESTS_PER_TOKEN} requests")

Token counters reset.
Fresh capacity today: 188 requests


In [9]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 10 — Full batch (FY_ANNUAL only)                   ║
# ╚══════════════════════════════════════════════════════════╝
#
# Processes only FY_ANNUAL PDFs from pdf_metadata table.
# H1_INTERIM records are completely skipped.
#
# RESUME-SAFE: records with conf >= 0.5 and needs_review=False
# are skipped. Run every day — picks up where stopped.
#
# Daily capacity with 4 tokens: 188 requests
# After Cell 9 uses 4-8: ~180-184 new PDFs processed per day
# ~400 FY_ANNUAL PDFs total → done in 2-3 days
 
def decode_date_from_filename(filename: str):
    """Decode period_end_date from filename when DB value is NULL."""
    name = filename.replace('.pdf', '').replace('.PDF', '')
    # Pattern: DDMMYY at end e.g. efd311224 → 2024-12-31
    m = re.search(r'(\d{2})(\d{2})(\d{2})$', name)
    if m:
        try:
            from datetime import date as ddate
            day = int(m.group(1))
            mon = int(m.group(2))
            yr  = int('20' + m.group(3))
            if 1 <= day <= 31 and 1 <= mon <= 12 and 2010 <= yr <= 2030:
                return str(ddate(yr, mon, day))
        except:
            pass
    # Pattern: DD-MM-YY
    m2 = re.search(r'(\d{2})-(\d{2})-(\d{2})', name)
    if m2:
        try:
            from datetime import date as ddate
            return str(ddate(int('20'+m2.group(3)),
                             int(m2.group(2)), int(m2.group(1))))
        except:
            pass
    # Pattern: 4-digit year only
    m3 = re.search(r'(20\d{2})', name)
    if m3:
        yr = int(m3.group(1))
        return f'{yr}-12-31'  # FY always ends Dec 31
    return None
 
 
def run_batch():
    """Process all FY_ANNUAL PDFs from pdf_metadata table."""
    local_isin_map = get_isin_map()
 
    # Load ONLY FY_ANNUAL records
    conn = get_conn()
    df = pd.read_sql('''
        SELECT id, ticker, filename, report_type,
               period, period_end_date, local_path
        FROM pdf_metadata
        WHERE download_status = 'downloaded'
          AND report_type = 'FY_ANNUAL'
        ORDER BY ticker, fiscal_year
    ''', conn)
    conn.close()
 
    total     = len(df)
    processed = 0
    skipped   = 0
    failed    = 0
    review    = 0
    retried   = 0
    stopped   = False
 
    total_limit = len(GITHUB_TOKENS) * REQUESTS_PER_TOKEN
    total_used  = sum(_token_requests)
 
    print("=" * 70)
    print(f"GPT-4o FY_ANNUAL batch — {total} PDFs")
    print(f"Tokens: {len(GITHUB_TOKENS)} accounts × "
          f"{REQUESTS_PER_TOKEN} req = {total_limit} max/day")
    print(f"Already used today: {total_used}")
    print(f"Remaining today: {total_limit - total_used}")
    print("=" * 70)
    print()
 
    for _, row in df.iterrows():
        if stopped:
            break
 
        seq = processed + skipped + failed + 1
 
        # ── Resolve PDF path ──────────────────────────────
        pdf_path = None
        if row['local_path']:
            pdf_path = Path(row['local_path'])
        if not pdf_path or not pdf_path.exists():
            safe     = re.sub(r'[<>:"/\\|?*]', '_', row['ticker'])
            pdf_path = (PDF_BASE_DIR / safe /
                        row['report_type'] / row['filename'])
        if not pdf_path.exists():
            print(f"[{seq:4d}/{total}] ✗ NOT FOUND: "
                  f"{row['ticker']} / {row['filename']}")
            failed += 1
            continue
 
        # ── Resolve period_end_date ───────────────────────
        ped     = row['period_end_date']
        ped_str = str(ped) if ped else ''
        if ped_str in ('', 'NaT', 'None', 'nan'):
            ped_str = decode_date_from_filename(row['filename']) or ''
        if not ped_str:
            print(f"[{seq:4d}/{total}] ✗ NULL DATE: {row['filename']}")
            failed += 1
            continue
 
        # ── Print progress line ───────────────────────────
        used = sum(_token_requests)
        print(f"[{seq:4d}/{total}] "
              f"{row['ticker']:<26} {row['filename']:<38} "
              f"[{used}/{total_limit}]",
              end=' ', flush=True)
 
        # ── Extract ───────────────────────────────────────
        status = extract_one_pdf(
            pdf_path,
            row['ticker'],
            row['period'],
            ped_str,
            local_isin_map
        )
 
        # ── Handle stop conditions ────────────────────────
        if status['notes'] == 'all_tokens_exhausted':
            print(f"\n\nAll {len(GITHUB_TOKENS)} tokens exhausted today.")
            print(f"Total requests used: {sum(_token_requests)}")
            print("Run again tomorrow — already-extracted records are safe.")
            stopped = True
            break
 
        if status['skipped']:
            skipped += 1
            print("↷ skipped")
            continue
 
        # ── Print result ──────────────────────────────────
        processed += 1
        if status['review']:
            review += 1
        if status['used_retry']:
            retried += 1
 
        icon       = '✓' if not status['review'] else '⚠'
        retry_flag = ' [R]' if status['used_retry'] else ''
        print(f"{icon} conf={status['confidence']:.2f}  "
              f"{status['notes'][:45]}{retry_flag}")
 
        time.sleep(DELAY_BETWEEN_PDFS)
 
        # Progress summary every 50 PDFs
        if seq % 50 == 0:
            print()
            print(f"  ── {seq}/{total} ── "
                  f"processed={processed} skipped={skipped} "
                  f"failed={failed} review={review} "
                  f"retried={retried} "
                  f"requests={sum(_token_requests)}/{total_limit}")
            print()
 
    # ── Final summary ─────────────────────────────────────
    if not stopped:
        print()
        print("=" * 70)
        print("BATCH COMPLETE")
        print(f"  Processed:     {processed:,}")
        print(f"  Skipped:       {skipped:,}  (already extracted)")
        print(f"  Failed:        {failed:,}  (file not found or no date)")
        print(f"  Needs review:  {review:,}  (low conf or balance error)")
        print(f"  Used 2-pass:   {retried:,}  (auto-corrected)")
        print(f"  API requests:  {sum(_token_requests)}/{total_limit}")
        print("=" * 70)
 
 
# Uncomment after Cell 9 passes:
run_batch()
 
 

GPT-4o FY_ANNUAL batch — 266 PDFs
Tokens: 4 accounts × 47 req = 188 max/day
Already used today: 1
Remaining today: 187

[   1/266] ADWYA                      adwya_efd311216.pdf                    [1/188] ↷ skipped
[   2/266] ADWYA                      adwya_efd311217.pdf                    [1/188] ↷ skipped
[   3/266] ADWYA                      adwya_efd311218.pdf                    [1/188] ↷ skipped
[   4/266] ADWYA                      adwya_efd_31122019.pdf                 [1/188] ↷ skipped
[   5/266] ADWYA                      adwya_efd311220.pdf                    [1/188] 

C:\Users\Negza\AppData\Local\Temp\ipykernel_3656\3631421546.py:53: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql('''


↷ skipped
[   6/266] ADWYA                      adwya_efd311221.pdf                    [1/188] ↷ skipped
[   7/266] ADWYA                      adwya_efd311222.pdf                    [1/188] ⚠ conf=0.00  total_assets:missing
[   8/266] AMEN BANK                  amen_bank_efd311216.pdf                [2/188] ↷ skipped
[   9/266] AMEN BANK                  amen_bank_efd311217.pdf                [2/188] ↷ skipped
[  10/266] AMEN BANK                  amen_bank_efd311218.pdf                [2/188] ↷ skipped
[  11/266] AMEN BANK                  efd_2019_amen_bank.pdf                 [2/188] ↷ skipped
[  12/266] AMEN BANK                  amen_bank_efd311220.pdf                [2/188] ↷ skipped
[  13/266] AMEN BANK                  amen_bank_efd311221.pdf                [2/188] ↷ skipped
[  14/266] AMEN BANK                  amen_bank_efd311222.pdf                [2/188] ↷ skipped
[  15/266] AMEN BANK                  amen_bank_efd311223.pdf                [2/188] ↷ skipped
[  16/266] AMEN 

In [13]:
import time
from datetime import datetime, timezone

# Check current UTC time
utc_now = datetime.now(timezone.utc)
print(f"Current UTC time: {utc_now.strftime('%H:%M:%S')}")
print(f"Your local time:  {datetime.now().strftime('%H:%M:%S')}")
print()
print("GitHub quota resets at 00:00 UTC")

# Quick test of token 1
try:
    test = get_client().chat.completions.create(
        model=GPT_MODEL,
        messages=[{"role": "user", "content": "Say READY"}],
        max_tokens=5
    )
    print(f"Token 1: AVAILABLE — {test.choices[0].message.content.strip()}")
except Exception as e:
    if '429' in str(e):
        print(f"Token 1: EXHAUSTED — quota not reset yet")
    else:
        print(f"Token 1: ERROR — {str(e)[:80]}")

Current UTC time: 00:56:15
Your local time:  01:56:15

GitHub quota resets at 00:00 UTC
Token 1: EXHAUSTED — quota not reset yet


In [10]:
# ╔══════════════════════════════════════════════════════════╗
# ║  CELL 11 — Verify results                                ║
# ╚══════════════════════════════════════════════════════════╝
 
conn = get_conn()
 
df_quality = pd.read_sql('''
    SELECT
        company_type,
        COUNT(*)                                               AS total,
        ROUND(AVG(extraction_confidence)::numeric, 3)         AS avg_conf,
        COUNT(*) FILTER (WHERE needs_review = FALSE)          AS clean,
        COUNT(*) FILTER (WHERE needs_review = TRUE)           AS review,
        COUNT(*) FILTER (WHERE total_assets IS NOT NULL)      AS has_assets,
        COUNT(*) FILTER (WHERE net_result   IS NOT NULL)      AS has_net,
        COUNT(*) FILTER (WHERE extraction_confidence >= 0.5)  AS high_conf
    FROM financial_statements
    GROUP BY company_type
    ORDER BY company_type
''', conn)
 
df_errors = pd.read_sql('''
    SELECT ticker, period, company_type,
           total_assets, net_result, equity,
           extraction_confidence, needs_review,
           extraction_notes
    FROM financial_statements
    WHERE needs_review = TRUE
       OR extraction_confidence < 0.4
    ORDER BY ticker, period
    LIMIT 30
''', conn)
 
df_sample = pd.read_sql('''
    SELECT ticker, period, total_assets, equity,
           net_result, pnb, revenue,
           extraction_confidence
    FROM financial_statements
    WHERE extraction_confidence >= 0.5
      AND needs_review = FALSE
    ORDER BY ticker, period
    LIMIT 20
''', conn)
 
conn.close()
 
print("Quality by company type:")
print(df_quality.to_string(index=False))
print()
print(f"Records needing review or low confidence ({len(df_errors)}):")
print(df_errors.to_string(index=False))
print()
print("Clean high-confidence records (first 20):")
print(df_sample.to_string(index=False))
 

Quality by company type:
company_type  total  avg_conf  clean  review  has_assets  has_net  high_conf
        bank     63     0.982     63       0          63       61         63
   insurance     12     0.967     11       1          11       12         12
     leasing     36     1.000     36       0          36       36         36
    non_bank    150     0.999    148       2         150      150        150

Records needing review or low confidence (3):
        ticker  period company_type  total_assets  net_result     equity  extraction_confidence  needs_review                                         extraction_notes
           AMS FY 2018     non_bank     36276.697  -19444.988 -39122.813                    1.0          True retry_no_improvement | net_gt_30pct_assets(-19445>36277)
BNA ASSURANCES FY 2024    insurance           NaN   16315.042 100920.836                    0.6          True                                     total_assets:missing
      SERVICOM FY 2019     non_bank      8

C:\Users\Negza\AppData\Local\Temp\ipykernel_3656\3109923128.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_quality = pd.read_sql('''
C:\Users\Negza\AppData\Local\Temp\ipykernel_3656\3109923128.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_errors = pd.read_sql('''
C:\Users\Negza\AppData\Local\Temp\ipykernel_3656\3109923128.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sample = pd.read_sql('''
